# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"Dataset name: {getattr(dataset.metadata, 'name', None)}\nDescription: {getattr(dataset.metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll examine the record sets available in the Croissant metadata and inspect their fields and field IDs. For each record set, we obtain its `@id` and enumerate the fields and columns by their `@id` as well.

In [ ]:
# Utility to extract record set ids from the dataset
record_sets = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
if not record_sets:
    # If recordSet is empty, probe from distributions
    print("No direct record sets referenced. Let's examine the distributions for tabular data sources.")
    for dist in dataset.metadata.to_json().get('distribution', []):
        obj_id = dist.get('@id', None)
        print(f"Distribution: {obj_id}")
else:
    print("Available record sets and their @id:")
    for rs in dataset.metadata.to_json()['recordSet']:
        print(f"- {rs['@id']}")

# List fields for each record set
for rs in dataset.metadata.to_json().get('recordSet', []):
    print(f"\nRecord set: {rs['@id']}")
    print("Fields and their @id:")
    for field in rs.get('field', []):
        if isinstance(field, dict) and '@id' in field:
            print(f"  - {field['@id']}")

# If no record sets in metadata, use a default example @id
print("\nFor this dataset, since the Croissant schema does not list explicit record sets, we will inspect the available records by loading.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

**Note:** If no `recordSet` is defined, use the default or main tabular data resource's `@id`, which can be found in the `distribution` list in the schema. We'll extract records using its `@id`.

In [ ]:
# 1. Obtain main record set or distribution id
json_md = dataset.metadata.to_json()
record_sets = [rs['@id'] for rs in json_md.get('recordSet', [])]
if record_sets:
    # Use first record set as main
    main_record_set_id = record_sets[0]
else:
    # Use first distribution as main
    distributions = json_md.get('distribution', [])
    main_record_set_id = distributions[0]['@id'] if len(distributions) > 0 else None

if main_record_set_id is None:
    raise ValueError("No record set or distribution @id found in metadata.")

# 2. List all distributions/record_set @ids
all_record_set_ids = record_sets if record_sets else [d['@id'] for d in json_md.get('distribution', [])]
dataframes = {}

for rsid in all_record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        if records:
            dataframes[rsid] = pd.DataFrame(records)
            print(f"Loaded records for record set/distribution @id: {rsid}, shape: {dataframes[rsid].shape}")
    except Exception as e:
        print(f"Error loading records for @id {rsid}: {e}")

# 3. Display columns and sample records from the main DataFrame
print(f"\nColumns in data for @id {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist() if main_record_set_id in dataframes else "(No data loaded)")
if main_record_set_id in dataframes:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

We will look for numeric fields (e.g. `mw`, `coverage_percent`, `abundance_*`), filter on one, normalize it, and (optionally) group by another field such as protein accession.

In [ ]:
# Choose a representative numeric field id, e.g., a column related to molecular weight ("MW")
df = dataframes[main_record_set_id]
possible_numeric_fields = [col for col in df.columns if ('percent' in col.lower() or 'mw' in col.lower() or col.lower().startswith('abundance') or 'count' in col.lower())]
if len(possible_numeric_fields) > 0:
    numeric_field = possible_numeric_fields[0]
else:
    # Default to the first numeric-looking column if none found
    numeric_field = df.select_dtypes(include='number').columns[0] if not df.empty and len(df.select_dtypes(include='number').columns) > 0 else df.columns[0]

print(f"Using numeric field: {numeric_field}")

# Filter: Only rows where the numeric_field is above threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field]) else df
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Add a normalized version of the numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Optional: Group by a likely categorical field (e.g., 'accession' or first object/text column)
group_field = None
for candidate in ['accession', 'protein_accession', 'id', 'description']:
    if candidate in df.columns:
        group_field = candidate
        break
if not group_field:
    # Try fallback: first object dtype
    for col in df.columns:
        if df[col].dtype == 'object' and col != numeric_field:
            group_field = col
            break

if group_field and group_field in filtered_df.columns and pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field and group_field in df.columns:
        # Boxplot by group if reasonable cardinality
        groups = df[group_field].value_counts()
        if groups.size < 50:
            plt.figure(figsize=(12, 5))
            sns.boxplot(x=group_field, y=numeric_field, data=df)
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we've demonstrated loading and exploring the FAIR² mass spectrometry dataset using `mlcroissant`. We inspected the schema, extracted tabular data using distribution `@id`s, and performed EDA illustrating numeric field filtering, normalization, groupwise examination, and basic visualizations. This workflow can be extended for further domain- and task-specific analytical steps.
